# Stage 04 - Model Evaluation with MLflow + DagsHub

> Security: a DagsHub access token was previously exposed. Revoke/rotate it before running this notebook; never store the replacement token here.


In [22]:
import os
import sys

print(sys.executable)


c:\Users\USER\Downloads\Kidney_diesease _classification\kidney\Scripts\python.exe


In [23]:
%pwd

'c:\\Users\\USER\\Downloads'

In [24]:
%pwd

'c:\\Users\\USER\\Downloads'

In [25]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/anu705545/Kidney-Disease-Classification-MLflow-DVC.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] ="anu70554"
os.environ["MLFLOW_TRACKING_PASSWORD"] ="9b7901a55ebaf841e82c6523e79cf6800673e56a"

In [26]:
import tensorflow as tf

In [28]:
from pathlib import Path
import os

PROJECT_ROOT = Path(r"C:\Users\USER\Downloads\Kidney_diesease _classification")

os.chdir(PROJECT_ROOT)

print("Current directory:")
print(Path.cwd())

MODEL_PATH = PROJECT_ROOT / "artifacts" / "training" / "model.h5"

print("\nModel path:")
print(MODEL_PATH)

print("\nModel exists:")
print(MODEL_PATH.exists())

Current directory:
C:\Users\USER\Downloads\Kidney_diesease _classification

Model path:
C:\Users\USER\Downloads\Kidney_diesease _classification\artifacts\training\model.h5

Model exists:
True


In [29]:
model = tf.keras.models.load_model(str(MODEL_PATH))

print("Model loaded successfully!")

Model loaded successfully!


In [31]:
model = tf.keras.models.load_model(
    "artifacts/training/model.h5"
)

## 1. Entity — EvaluationConfig

In [32]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

## 2. Configuration Manager

In [33]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri="https://dagshub.com/anu705545/Kidney-Disease-Classification-MLflow-DVC.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

## 3. Evaluation Component

In [34]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1. / 255,
            validation_split=0.30
        )
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run(run_name="VGG16Model-evaluation"):
            mlflow.log_params(self.config.all_params)
            mlflow.set_tags({
                "model_name": "VGG16Model",
                "evaluation_stage": "evaluation",
                "image_size": str(self.config.params_image_size),
                "batch_size": str(self.config.params_batch_size),
                "evaluation_dataset": str(self.config.training_data),
                "evaluation_split": "validation",
            })
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            mlflow.log_artifact("scores.json")

            # Model registry does not work with file store.
            if tracking_url_type_store != "file":
                mlflow.keras.log_model(
                    self.model, "model", registered_model_name="VGG16Model"
                )
            else:
                mlflow.keras.log_model(self.model, "model")


## 4. DagsHub + MLflow Setup

In [35]:
import dagshub
dagshub.init(repo_owner='anu705545', repo_name='Kidney-Disease-Classification-MLflow-DVC', mlflow=True)

[2026-10-06 21:44:59,668: INFO: _client]: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


Accessing as anu705545

[2026-10-06 21:44:59,691: INFO: helpers]: Accessing as anu705545
[2026-10-06 21:45:02,576: INFO: _client]: HTTP Request: GET https://dagshub.com/api/v1/repos/anu705545/Kidney-Disease-Classification-MLflow-DVC "HTTP/1.1 200 OK"
[2026-10-06 21:45:04,633: INFO: _client]: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


Initialized MLflow to track repo "anu705545/Kidney-Disease-Classification-MLflow-DVC"

[2026-10-06 21:45:06,785: INFO: helpers]: Initialized MLflow to track repo "anu705545/Kidney-Disease-Classification-MLflow-DVC"


Repository anu705545/Kidney-Disease-Classification-MLflow-DVC initialized!

[2026-10-06 21:45:06,785: INFO: helpers]: Repository anu705545/Kidney-Disease-Classification-MLflow-DVC initialized!


## 5. Run Evaluation Pipeline

In [36]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-10-06 21:45:18,212: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-10-06 21:45:18,215: INFO: common]: yaml file: params.yaml loaded successfully
[2026-10-06 21:45:18,216: INFO: common]: Directory created at: artifacts
Found 139 images belonging to 2 classes.
9/9 [==============================] - 19s 2s/step - loss: 4.1105e-05 - accuracy: 1.0000
[2026-10-06 21:45:37,674: INFO: common]: json file saved at: scores.json


2026/10/06 21:45:44 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-10-06 21:45:45,667: WARNING: save]: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.
INFO:tensorflow:Assets written to: C:\Users\USER\AppData\Local\Temp\tmpm22zm3fo\model\data\model\assets
[2026-10-06 21:45:46,383: INFO: builder_impl]: Assets written to: C:\Users\USER\AppData\Local\Temp\tmpm22zm3fo\model\data\model\assets


c:\Users\USER\Downloads\Kidney_diesease _classification\kidney\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/10/06 21:50:18 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 5
Created version '5' of model 'VGG16Model'.
